## 2.7 文本预处理 - 从原始 batch 文本到可送入 RNN 的 PyTorch 完整示例

#### 1、这一小节我们要做什么

##### 1.1 这一节的目标
这一节我们不再只停留在概念层面，而是直接通过一个完整的小案例，把前面学过的所有关键步骤真正串起来。

我们要演示的完整链路是：

原始 batch 文本  
$\rightarrow$ 分词  
$\rightarrow$ 建立词表  
$\rightarrow$ token 转 id  
$\rightarrow$ 统计有效长度 `lengths`  
$\rightarrow$ padding 成统一长度  
$\rightarrow$ 送入 Embedding  
$\rightarrow$ 得到可输入 RNN 的张量  
$\rightarrow$ 可选：进一步 pack，供 RNN 更高效处理

##### 1.2 这一节的意义
前面几节我们已经分别学过：

- 文本为什么不能直接输入 RNN
- token 和 tokenization
- vocabulary 和 token 到数字的映射
- one-hot 与 Embedding 的区别
- padding、有效长度、mask、pack 的概念

但是如果这些知识点彼此分散，就很容易停留在“每个点都懂一点，但整体流程不清楚”的状态。

所以这一节非常重要，因为它会把前面的知识真正落地成一条完整的 PyTorch 处理流程。

##### 1.3 这一节的重点
这一节你需要特别关注两件事：

第一，**数据类型在每一步到底发生了什么变化**。  
第二，**张量形状在每一步到底是怎么变化的**。

因为文本预处理最容易混乱的地方，往往不是概念本身，而是：

- 现在这个变量到底是字符串、列表、id 还是张量
- 当前 shape 到底是 `(seq_len,)`、`(batch_size, seq_len)` 还是 `(batch_size, seq_len, embedding_dim)`

所以这一节本质上是在帮助你建立一个非常清晰的“数据流转视角”。

#### 2、PyTorch 中这一整套流程涉及到的关键组件

##### 2.1 `nn.Embedding`
在 PyTorch 里：

```python
nn.Embedding
```

本质上是一个“根据索引取词向量”的查表层。

它的输入是整数 id 张量，输出是对应的词向量张量。

也就是说：

- 输入：token 的编号
- 输出：每个 token 对应的 embedding 向量

##### 2.2 `pad_sequence`
在 PyTorch 里：

```python
pad_sequence
```

可以把不同长度的序列自动补齐成统一长度的张量。

当我们设置：

```python
batch_first=True
```

时，输出形状就是：

- `B × T`
- 或 `B × T × *`

其中：

- `B` 表示 `batch_size`
- `T` 表示补齐后的统一序列长度

##### 2.3 `nn.RNN`
在 PyTorch 中，如果：

```python
batch_first=True
```

那么 `nn.RNN` 的输入形状是：

$N \times L \times H_{in}$

其中：

- $N$：batch size
- $L$：sequence length
- $H_{in}$：每个时间步输入向量维度

它还支持接收 `PackedSequence`，也就是支持变长序列的更高效处理方式。


#### 三、步骤 1：准备原始 batch 文本

##### 3.1 代码
```python
sentences = [
    "i love ai",
    "pytorch is great",
    "i love deep learning"
]

print("原始 batch 文本：")
for s in sentences:
    print(s)
```

##### 3.2 解释
这一步的数据还是最原始的 Python 字符串列表。

现在的数据类型本质上是：

```python
list[str]
```

也就是说，它还只是“人能读懂的文本”，并不是模型可以直接处理的数值数据。

##### 3.3 为什么这里是 batch 文本
这里的 `sentences` 不是一句话，而是一个 batch 中的多条句子。

例如：

- `"i love ai"`
- `"pytorch is great"`
- `"i love deep learning"`

这表示：

当前我们不是在处理单条文本，而是在模拟训练时一个小批次的输入。

这点很重要，因为后面我们要做的 padding、mask、Embedding，都是围绕 batch 展开的。


#### 4、步骤 2：分词

##### 4.1 代码
```python
tokenized_sentences = []

for s in sentences:
    tokens = s.lower().split()
    tokenized_sentences.append(tokens)

print("分词结果：")
for tokens in tokenized_sentences:
    print(tokens)
```

##### 4.2 讲解
这里为了演示最基础流程，我们直接使用：

```python
lower() + split()
```

也就是：

- 先转小写
- 再按空格切分

例如：

```python
"I love ai"
```

会变成：

```python
["i", "love", "ai"]
```

##### 4.3 为什么 `append` 进去的是一整条 token 列表
这里要特别注意：

```python
split()
```

返回的不是单个 token，而是一个列表。

所以：

```python
tokens = s.lower().split()
```

得到的是一整条句子的 token 列表，例如：

```python
["i", "love", "ai"]
```

然后：

```python
tokenized_sentences.append(tokens)
```

添加进去的也是这整条句子的 token 列表。

因此最后的结果是：

```python
list[list[str]]
```

也就是说：

- 外层列表表示一个 batch
- 内层每个列表表示一条句子的 token 序列

##### 4.4 这一步后的数据类型
这一步后，数据类型变成：

```python
list[list[str]]
```

例如：

```python
[
    ["i", "love", "ai"],
    ["pytorch", "is", "great"],
    ["i", "love", "deep", "learning"]
]
```

这说明：

我们已经从“原始整句字符串”，进入到了“token 序列”的阶段。


#### 5、步骤 3：建立词表 vocabulary

##### 5.1 代码
```python
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for tokens in tokenized_sentences:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

print("词表 vocab：")
for token, idx in vocab.items():
    print(f"{token:>10} -> {idx}")
```

##### 5.2 讲解
这一步是在建立：

**token 到编号的统一规则**

也就是构建 vocabulary。

这里我们先手动加入两个特殊 token：

- `<PAD>`：用于后面补齐长度
- `<UNK>`：用于处理未知词

##### 5.3 外循环和内循环分别在做什么
这里有两层循环：

```python
for tokens in tokenized_sentences:
```

这一层表示：

依次取出 batch 中的每一条句子的 token 列表。

然后：

```python
for token in tokens:
```

这一层表示：

再去遍历这条句子中的每一个 token。

也就是说：

- 外循环遍历句子
- 内循环遍历句子里的每个 token

##### 5.4 为什么使用 `len(vocab)` 来赋值
这句代码：

```python
vocab[token] = len(vocab)
```

的意思是：

如果当前 token 不在词表中，就给它分配一个新的编号，而这个编号正好等于当前词表长度。

例如一开始：

```python
{
    "<PAD>": 0,
    "<UNK>": 1
}
```

这时 `len(vocab) = 2`，所以第一个新单词会得到编号 `2`。

这样做的好处是：

每加入一个新 token，就自动给它一个不重复的新 id。

##### 5.5 一个可能的最终词表
例如最后可能得到：

```python
<PAD> -> 0
<UNK> -> 1
i -> 2
love -> 3
ai -> 4
pytorch -> 5
is -> 6
great -> 7
deep -> 8
learning -> 9
```

#### 6、步骤 4：把 token 转成 id 序列

##### 6.1 代码
```python
def tokens_to_ids(tokens, vocab):
    unk_id = vocab["<UNK>"]
    return [vocab.get(token, unk_id) for token in tokens]

id_sequences = [tokens_to_ids(tokens, vocab) for tokens in tokenized_sentences]

print("token 转 id 后：")
for ids in id_sequences:
    print(ids)
```

##### 6.2 讲解
这一步是真正把文本变成数字。

例如：

```python
["i", "love", "ai"]
```

会变成：

```python
[2, 3, 4]
```

这说明：

文本已经不再是字符串序列，而是整数编号序列。

##### 6.3 `vocab.get(token, unk_id)` 的作用
这里使用的是：

```python
vocab.get(token, unk_id)
```

意思是：

- 如果 token 在词表中，就返回它的 id
- 如果不在词表中，就返回 `<UNK>` 的 id

这一步是为了处理未知词问题。

虽然这个示例中的文本都来自当前词表，几乎不会真的触发 `<UNK>`，但这种写法更接近真实项目的处理方式。

##### 6.4 这一步后的数据类型
现在的数据类型可以理解为：

```python
list[list[int]]
```

例如：

```python
[
    [2, 3, 4],
    [5, 6, 7],
    [2, 3, 8, 9]
]
```

##### 6.5 为什么这时还不能直接送进 RNN
虽然文本已经变成数字了，但这时仍然不能直接送进 RNN，因为：

- 它还只是普通 Python 列表
- 不同句子长度还不统一
- 还没有变成张量
- 还没有经过 Embedding

所以这一步只是完成了“编号化”，还没有完成“张量化”和“向量化”。

#### 7、步骤 5：把每条 id 序列转成 tensor，并记录有效长度 `lengths`

##### 7.1 代码
```python
import torch

sequence_tensors = [torch.tensor(ids, dtype=torch.long) for ids in id_sequences]
lengths = torch.tensor([len(seq) for seq in sequence_tensors], dtype=torch.long)

print("每条序列对应的 tensor：")
for seq in sequence_tensors:
    print(seq)

print("有效长度 lengths：")
print(lengths)
```

##### 7.2 第一个重点：先把每条序列转成 `torch.tensor`
这里我们把每条 id 序列都转成了张量：

```python
torch.tensor(ids, dtype=torch.long)
```

例如：

```python
[2, 3, 4]
```

会变成：

```python
tensor([2, 3, 4])
```

##### 7.3 为什么 `dtype` 要用 `torch.long`
这点非常重要。

因为后面 Embedding 层接收的是：

**整数索引张量**

而不是浮点张量。

所以这里必须写成：

```python
dtype=torch.long
```

如果写成 float，后面传给 `nn.Embedding` 时就会报错。

##### 7.4 第二个重点：记录有效长度 `lengths`
这句代码：

```python
lengths = torch.tensor([len(seq) for seq in sequence_tensors], dtype=torch.long)
```

的作用是：

记录每条句子的真实长度，也就是有效长度。

例如：

```python
[2, 3, 4]       # 长度 3
[5, 6, 7]       # 长度 3
[2, 3, 8, 9]    # 长度 4
```

那么：

```python
lengths = [3, 3, 4]
```

##### 7.5 为什么 `lengths` 很重要
后面如果你要使用：

```python
pack_padded_sequence
```

那么这个 `lengths` 会非常关键。

因为 pack 的核心就是利用每条序列的真实长度，让 RNN 尽量只处理有效 token，而不是把 padding 位置也一起算进去。

##### 7.6 这一步后的数据状态
现在每条句子已经变成了：

- 一维 LongTensor
- 每条 shape 是 `(seq_len,)`

但由于长度还不同，仍然不能直接拼成规则 batch 张量。

所以接下来还要做 padding。

#### 8、步骤 6：padding 成统一长度

##### 8.1 代码
```python
from torch.nn.utils.rnn import pad_sequence

pad_id = vocab["<PAD>"]

padded_ids = pad_sequence(
    sequence_tensors,
    batch_first=True,
    padding_value=pad_id
)

print("padding 后的 id 张量：")
print(padded_ids)
print("shape =", padded_ids.shape)
```

##### 8.2 讲解
不同句子长度不同，不能直接组成规则 batch，所以要先补齐。

```python
pad_sequence
```

会自动找到当前 batch 中最长的序列长度，然后把其他短序列补齐到同样长度。

##### 8.3 `batch_first=True` 的意义
当设置：

```python
batch_first=True
```

时，输出形状就是 batch 在前。

也就是说：

```python
(batch_size, seq_len)
```

而不是：

```python
(seq_len, batch_size)
```

这和我们后面设置 `nn.RNN(batch_first=True)` 是对应起来的，更容易理解。

##### 8.4 `padding_value=pad_id` 的意义
这里我们规定：

```python
pad_id = vocab["<PAD>"]
```

也就是：

`<PAD>` 的 id 用来补齐空位。

所以补齐时，短句后面会自动填充 `0`（如果 `<PAD> = 0`）。

##### 8.5 一个可能的 padding 结果
假设最长句子长度是 `4`，那么结果可能是：

```python
[
    [2, 3, 4, 0],
    [5, 6, 7, 0],
    [2, 3, 8, 9]
]
```

这时 shape 就是：

```python
(3, 4)
```

也就是：

- `batch_size = 3`
- `seq_len = 4`

##### 8.6 这一步后的张量类型
这里的数据类型仍然是 `LongTensor`，因为它本质上仍然只是 token 的 id。

也就是说：

padding 后虽然变成了规则 batch 张量，但它还不是词向量张量。

#### 9、步骤 7：生成 mask

##### 9.1 代码
```python
mask = (padded_ids != pad_id).long()

print("mask：")
print(mask)
print("shape =", mask.shape)
```

##### 9.2 讲解
mask 的作用是标记：

- 哪些位置是真实 token
- 哪些位置是 `PAD`

这里的写法：

```python
(padded_ids != pad_id)
```

会先生成一个布尔张量：

- 不是 `PAD` 的位置为 `True`
- 是 `PAD` 的位置为 `False`

然后：

```python
.long()
```

再把它转换成整数：

- `True -> 1`
- `False -> 0`

##### 9.3 一个具体例子
如果：

```python
padded_ids =
[
    [2, 3, 4, 0],
    [5, 6, 7, 0],
    [2, 3, 8, 9]
]
```

那么：

```python
mask =
[
    [1, 1, 1, 0],
    [1, 1, 1, 0],
    [1, 1, 1, 1]
]
```

##### 9.4 mask 的意义
这里：

- `1` 表示有效位置
- `0` 表示 padding 位置

虽然这个例子后面主要展示 RNN 输入，但 mask 这个概念很重要，因为后面在做：

- loss 计算
- 池化
- attention
- 有效位置筛选

时经常会用到它。

#### 10、步骤 8：定义 Embedding，并把 id 张量变成词向量张量

##### 10.1 代码
```python
import torch.nn as nn

vocab_size = len(vocab)
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)

embedded = embedding(padded_ids)

print("Embedding 后的张量：")
print("shape =", embedded.shape)
print("dtype =", embedded.dtype)
```

##### 10.2 最关键的一句
这一句最关键：

```python
embedded = embedding(padded_ids)
```

它的意思是：

把每个 token id 查表映射成一个 embedding 向量。

##### 10.3 Embedding 层内部本质上是什么
假设：

- 词表大小 `vocab_size = 10`
- `embedding_dim = 8`

那么 Embedding 层内部本质上可以看成一个：

$10 \times 8$

的矩阵。

其中：

- 一共有 `10` 行，对应 `10` 个 token
- 每一行是这个 token 的 `8` 维向量表示

##### 10.4 为什么输入是 `(3, 4)`，输出会变成 `(3, 4, 8)`
如果 `padded_ids` 的 shape 是：

```python
(3, 4)
```

表示：

- `3` 条句子
- 每条句子 `4` 个 token

那么经过 Embedding 后，每个 id 都会被替换成一个 `8` 维向量，所以 shape 会变成：

```python
(3, 4, 8)
```

也就是：

$batch\_size \times seq\_len \times embedding\_dim$

##### 10.5 `padding_idx=pad_id` 的意义
这里设置：

```python
padding_idx=pad_id
```

是为了告诉 Embedding：

这个 id 是专门用来表示 padding 的。

这样做通常有助于减少 padding 对训练的干扰。

##### 10.6 这一步后的数据状态
到了这一步，数据终于变成了真正适合送入 RNN 的 float 张量。

也就是说：

前面的 id 张量只是“编号”，  
而现在的 `embedded` 才是真正的“向量输入”。

#### 11、完整代码及步骤总结 🧠

##### 11.1 完整代码

In [1]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# 1. 构建原始文本数据
sentences = [
    "i love AI", 
    "pytorch is great",
    "i love deep learning"
]
print("原始文本数据:", sentences)


# 2. tokenization 分词操作
tokenized_sentences = []
for s in sentences:
    tokens = s.lower().split() # 简单的分词方法，注意 split 方法返回的是一个列表
    tokenized_sentences.append(tokens) # 将分词之后的列表结果添加到 tokenized_sentences 中，形成list[list[str]] 
for tokens in tokenized_sentences:
    print("分词结果:", tokens)


# 3. 建立词表 vocabulary
# 词表本质上是一个字典，起对照作用，key 是单词，value 是对应的索引
vocab = {
    "<PAD>": 0, # 用于填充的特殊标记
    "<UNK>": 1  # 用于未知单词的特殊标记
}
for tokens in tokenized_sentences:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab) # 将新单词添加到词表中，索引为当前词表的长度
print("词表:", vocab)

# 4. 把 token 转成 id 序列
def tokens_to_ids(tokens, vocab):
    unk_id = vocab["<UNK>"] # 获取未知单词的 id
    return [vocab.get(token, unk_id) for token in tokens] # 使用 vocab.get(token, unk_id) 来获取 token 的 id，如果 token 不在 vocab 中，则返回 unk_id

id_sequences = [tokens_to_ids(tokens, vocab) for tokens in tokenized_sentences]

print("token 转 id 后：")
for ids in id_sequences:
    print(ids)


# 5. 把每条 id 序列转成 tensor，并记录有效长度 lengths
sequence_tensors = [torch.tensor(ids, dtype=torch.long) for ids in id_sequences]
lengths = torch.tensor([len(seq) for seq in sequence_tensors], dtype=torch.long)

print("每条序列对应的 tensor：")
for seq in sequence_tensors:
    print(seq)

print("有效长度 lengths：")
print(lengths)

# 6. padding 成统一长度
pad_id = vocab["<PAD>"]
padded_ids = pad_sequence(
    sequence_tensors,
    batch_first=True,
    padding_value=pad_id
)

print("padding 后的 id 张量：")
print(padded_ids)
print("shape =", padded_ids.shape)


# 7. 生成 mask
mask = (padded_ids != pad_id).long()

print("mask：")
print(mask)
print("shape =", mask.shape)


# 8. 定义 Embedding，并把 id 张量变成词向量张量
vocab_size = len(vocab)
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)

embedded = embedding(padded_ids)

print("Embedding 后的张量：")
print("shape =", embedded.shape)
print("dtype =", embedded.dtype)

原始文本数据: ['i love AI', 'pytorch is great', 'i love deep learning']
分词结果: ['i', 'love', 'ai']
分词结果: ['pytorch', 'is', 'great']
分词结果: ['i', 'love', 'deep', 'learning']
词表: {'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'ai': 4, 'pytorch': 5, 'is': 6, 'great': 7, 'deep': 8, 'learning': 9}
token 转 id 后：
[2, 3, 4]
[5, 6, 7]
[2, 3, 8, 9]
每条序列对应的 tensor：
tensor([2, 3, 4])
tensor([5, 6, 7])
tensor([2, 3, 8, 9])
有效长度 lengths：
tensor([3, 3, 4])
padding 后的 id 张量：
tensor([[2, 3, 4, 0],
        [5, 6, 7, 0],
        [2, 3, 8, 9]])
shape = torch.Size([3, 4])
mask：
tensor([[1, 1, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]])
shape = torch.Size([3, 4])
Embedding 后的张量：
shape = torch.Size([3, 4, 8])
dtype = torch.float32


##### 11.2 数据流转总结
**步骤 1：原始文本**

```python
list[str]
```

**步骤 2：分词后**

```python
list[list[str]]
```

**步骤 3：token 转 id 后**

```python
list[list[int]]
```

**步骤 4：每条句子转 tensor**

- shape: `(seq_len,)`
- dtype: `long`

**步骤 5：padding 后**

- shape: `(batch_size, seq_len)`
- dtype: `long`

**步骤 6：Embedding 后**

- shape: `(batch_size, seq_len, embedding_dim)`
- dtype: `float`

**步骤 7：送入 RNN 后**

- `output shape: (batch_size, seq_len, hidden_size)`
- `h_n shape: (num_layers, batch_size, hidden_size)`
